In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json

from pathlib import Path

%matplotlib inline
plt.style.use("default")

In [ ]:
BASE_DATA_PATH = Path("../data/processed/weather_hourly_clean.csv")
ENRICHED_DATA_PATH = Path("../data/processed/weather_hourly_clean_enriched.csv")

DATA_PATH = ENRICHED_DATA_PATH if ENRICHED_DATA_PATH.exists() else BASE_DATA_PATH

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"])

df = df.sort_values("datetime").reset_index(drop=True)

print("Dataset loaded from:", DATA_PATH)

df.head()


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(df["datetime"], df["precip"], alpha=0.6)
plt.title("Hourly Rainfall (mm)")
plt.ylabel("mm")
plt.xlabel("Time")
plt.show()

In [ ]:
monthly = (
    df.set_index("datetime")["precip"]
      .resample("M")
      .sum()
)

plt.figure(figsize=(10,4))
monthly.plot()
plt.title("Monthly Rainfall")
plt.ylabel("mm")
plt.show()

In [ ]:
MODEL_PATH = Path("../models/hgb_D_next_6h.pkl")
META_PATH = Path("../models/hgb_D_next_6h_meta.json")

model = joblib.load(MODEL_PATH)
meta = json.loads(META_PATH.read_text())

FEATURES = meta["features"]
THRESHOLD = meta["threshold"]

THRESHOLD

In [ ]:
h = 6

df["target_6h"] = (
    pd.concat(
        [df["rain_1h"].shift(-i) for i in range(1, h+1)],
        axis=1
    )
    .max(axis=1)
)

df = df.dropna(subset=["target_6h"]).copy()
df["target_6h"] = df["target_6h"].astype(int)

In [ ]:
missing_features = [f for f in FEATURES if f not in df.columns]
for f in missing_features:
    df[f] = np.nan

X = df[FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0.0)

if missing_features:
    print("Missing model features in dataset; filled with 0.0:", missing_features)

df["prob_6h"] = model.predict_proba(X)[:,1]
df["pred_6h"] = (df["prob_6h"] >= THRESHOLD).astype(int)

df[["datetime","prob_6h","pred_6h","target_6h"]].head()


In [ ]:
plt.figure(figsize=(14,4))

plt.plot(df["datetime"], df["prob_6h"], label="Predicted prob", alpha=0.7)

plt.axhline(THRESHOLD, color="red", linestyle="--", label="Threshold")

plt.scatter(
    df.loc[df["target_6h"]==1,"datetime"],
    df.loc[df["target_6h"]==1,"prob_6h"],
    color="black",
    s=10,
    label="Actual rain"
)

plt.legend()
plt.title("6h Rain Prediction Probability")
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    df["target_6h"],
    df["pred_6h"],
    display_labels=["No rain","Rain"]
)

plt.title("6h Forecast Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

p, r, _ = precision_recall_curve(
    df["target_6h"],
    df["prob_6h"]
)

plt.figure(figsize=(6,5))
plt.plot(r, p)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve (6h)")
plt.grid(True)
plt.show()